# First mushroom model comparison

**Contributors:** initial scaffold and local verification prepared with AI assistance. Team reviewers and substantive changes will be recorded after review. See [the assistance record](../docs/AI_USE.md).

Compare a majority-class reference, logistic regression and random forest. Positive class `p` is encoded as 1. Preprocessing learns only from training data.

## Run the development experiment

The majority-class dummy is the reference. Logistic regression supplies a simple baseline. Random forest can capture nonlinear interactions. The random forest settings are an initial choice, not a completed tuning exercise. The output reports validation performance at threshold 0.5. No final test labels are evaluated.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src' / 'mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mushrooms import load_data, experiment
results = experiment(load_data())
display(results.round(4))

              model      split  threshold  accuracy  poison_precision  poison_recall     f1  roc_auc  average_precision  tn  fp  fn  tp  fit_seconds  train_accuracy
  majority_baseline validation        0.5     0.621            0.0000         0.0000 0.0000   0.5000              0.379 621   0 379   0        0.022          0.6213
logistic_regression validation        0.5     0.692            0.6802         0.3536 0.4653   0.6776              0.636 558  63 245 134        0.031          0.7130
      random_forest validation        0.5     0.751            0.8457         0.4195 0.5608   0.8016              0.760 592  29 220 159        0.449          0.8937

Rows: training=3000, validation=1000, reserved test=1000
Positive class: p (poisonous/unsafe). Results are validation metrics.


,model,split,threshold,accuracy,poison_precision,poison_recall,f1,roc_auc,average_precision,tn,fp,fn,tp,fit_seconds,train_accuracy
0,majority_baseline,validation,0.5,0.621,0.0000,0.0000,0.0000,0.5000,0.379,621,0,379,0,0.022,0.6213
1,logistic_regression,validation,0.5,0.692,0.6802,0.3536,0.4653,0.6776,0.636,558,63,245,134,0.031,0.7130
2,random_forest,validation,0.5,0.751,0.8457,0.4195,0.5608,0.8016,0.760,592,29,220,159,0.449,0.8937


## Look at errors, not only the ranking

For the positive class `p`, a false negative is a poisonous/unsafe example predicted edible. Inspect several examples and form a specific hypothesis about missing values, rare categories or threshold choice. A probability score has not been calibrated into a reliable real-world confidence.

In [2]:
import pandas as pd
errors = pd.read_csv(ROOT/'reports/mushroom_validation_errors.csv')
print('Validation errors in the RF pilot:', len(errors))
display(errors.head(10))

Validation errors in the RF pilot: 249


,cap-diameter,stem-height,stem-width,spore-print-color,gill-color,habitat,season,ring-type,cap-shape,stem-surface,jumbled_noise_0,jumbled_noise_1,class,poison_probability,predicted_class
0,NaN,6.42,13.51,NaN,NaN,d,u,g,x,NaN,x,c,p,0.232330,e
1,NaN,NaN,6.61,NaN,f,NaN,NaN,f,x,NaN,NaN,x,p,0.289268,e
2,9.01,8.19,19.25,NaN,n,d,a,p,x,NaN,x,s,p,0.271080,e
3,9.06,7.03,35.75,NaN,w,m,s,f,f,NaN,NaN,x,p,0.171605,e
4,7.80,8.17,18.84,NaN,NaN,d,a,f,x,NaN,f,s,p,0.442916,e
5,4.33,6.29,8.36,NaN,o,d,w,f,f,t,s,x,p,0.214606,e
6,14.30,6.74,45.10,NaN,NaN,d,w,f,f,NaN,s,b,p,0.303075,e
7,7.38,3.28,8.86,NaN,p,d,NaN,NaN,s,NaN,s,x,e,0.507937,p
8,NaN,4.33,3.46,NaN,w,NaN,u,e,b,NaN,f,f,p,0.431765,e
9,1.07,3.93,5.78,NaN,f,NaN,a,f,NaN,NaN,b,NaN,p,0.431577,e


## Next experiments

These are untuned development results. Investigate the missed poisonous examples, threshold choice and the contribution of noisy or highly missing features. Run an automated comparison, tune shortlisted models and preserve AWS evidence before evaluating the reserved test. The [README](../README.md) lists the remaining project tasks.

To explore the saved forest, run `python app/server.py` from the repository root. The interface is a local demonstration.